# VCA system identification

**System:** Magnetic Innovations MMA 240-380 moving magnet voice coil actuator with magnetic end springs (Vibrotwist project)

**Goal:** a nonlinear grey box model that simulates position and acceleration accurately over the full stroke, with physically meaningful parameters, ready for controller design.

This notebook currently contains **the plan only**. Each section describes what will be done, how, why, and what it produces. Implementation is added section by section.

The work is organized in five parts. Each part produces something the next one needs.

| Part | Name | Purpose |
|---|---|---|
| A | Modelling | Define the model structure, its parameters, and a simulator |
| B | Experiment design | Design the input signals and check them in simulation |
| C | Measurement | Run the experiments and prepare the data |
| D | Identification | Estimate the parameters in a sequence of steps |
| E | Validation and use | Test the model on unseen data and export it |

## Notebook outline

| Section | Content | Part | Needs hardware data |
|---|---|---|---|
| 0 | Outline, variables, settings | | no |
| 1 | Physical model and parameterization | A | no |
| 2 | Simulator | A | no |
| 3 | Signal design and simulated pre-check | B | no |
| 4 | Experiments and data loading | C | yes |
| 5 | Sensor calibration | D | yes, E0 |
| 6 | Static identification | D | yes, E1 |
| 7 | Local linear identification | D | yes, E2 |
| 8 | First complete model | D | yes |
| 9 | Stage 1, equation error least squares | D | yes, E3 |
| 10 | Stage 2, output error refinement | D | yes, E3 |
| 11 | Validation | E | yes, E4 |
| 12 | Results and export | E | yes |

Sections 0 to 3 can be built now, without new measurements. The existing 6.5 A chirp can serve as stand in data while sections 9 and 10 are developed, but its final role is validation (section 11).

## 0. Variables and known values

### Variables

| Symbol | Meaning | Unit |
|---|---|---|
| $x$ | Position of the moving mass | m |
| $\dot x$ | Velocity | m/s |
| $\ddot x$ | Acceleration | m/s$^2$ |
| $i$ | Coil current (measured), the model input | A |
| $m$ | Moving mass | kg |
| $K_f(x)$ | Motor constant, position dependent | N/A |
| $c$ | Viscous (eddy current) damping | N s/m |
| $F_s(x)$ | Magnetic spring force, position dependent | N |
| $x_0$ | Equilibrium position at zero current | m |
| $g_a$ | Accelerometer gain | V/(m/s$^2$) |
| $\sigma_x,\ \sigma_a$ | Noise levels of position and acceleration | m, m/s$^2$ |
| $\xi$ | State vector $[x,\ \dot x]^\top$ | |
| $\theta$ | Parameter vector | |
| $\varphi(k),\ \Phi,\ Y$ | Regressor, regressor matrix, output vector (stage 1) | |
| $k_{\text{eff}}(\bar x)$ | Local effective stiffness around position $\bar x$ | N/m |
| $\bar x,\ \bar i$ | Operating point position and DC current | m, A |
| $T_s$ | Sample time, $1/2000$ s | s |

### Known values (datasheet and earlier measurements)

| Quantity | Value | Status |
|---|---|---|
| $m$ | 51.5 kg | certain |
| $K_f(z)$ | $283 - 0.0198\,z - 0.578\,z^2$ N/A, $z$ in mm | datasheet, high frequency estimate 258 N/A |
| $c$ | 1160 N s/m | datasheet |
| $R$, $L$ | 4.75 $\Omega$, 16.5 mH | not needed, current controlled |
| Stroke | $\pm 18$ mm nominal, $\pm 20$ to $22$ mm measured | |
| Small signal resonance | about 4.5 Hz | from chirp |
| Sample rate | 2 kHz | known and stable |
| $g_a$ | unresolved | to be determined in section 5 |

# Part A. Modelling

## 1. Physical model and parameterization

### 1.1 System definition and assumptions
- One degree of freedom: the mass moves along a single axis.
- Rigid mount: the stator does not move, so frame dynamics and the disturbance $d$ are left out for now.
- Current controlled: the drive sets the current, so the electrical dynamics ($R$, $L$, back EMF) drop out and $i$ is the input.
- The measured current is used everywhere, never the target current, because tracking drops by about 13% near 50 Hz.

### 1.2 Equation of motion

$$m\,\ddot x = K_f(x)\,i \;-\; c\,\dot x \;+\; F_s(x)$$

### 1.3 Nonlinear state space form
States $\xi_1 = x$, $\xi_2 = \dot x$, input $u = i$:

$$\dot \xi = f(\xi, u) = \begin{bmatrix} \xi_2 \\ \dfrac{1}{m}\Big[K_f(\xi_1)\,u - c\,\xi_2 + F_s(\xi_1)\Big] \end{bmatrix}$$

Outputs, position and acceleration:

$$y = h(\xi, u) = \begin{bmatrix} \xi_1 \\ \dfrac{1}{m}\Big[K_f(\xi_1)\,u - c\,\xi_2 + F_s(\xi_1)\Big] \end{bmatrix}$$

### 1.4 Parameterization
Polynomial forms (orders to be confirmed by the data):

$$K_f(x) = k_0 + k_1 x + k_2 x^2, \qquad F_s(x) = a_0 + a_1 x + a_2 x^2 + a_3 x^3$$

The constant $a_0$ absorbs the offset $x_0$, which keeps the model linear in the parameters.

### 1.5 Regression form

$$m\,\ddot x(k) = \varphi(k)^\top \theta$$

$$\varphi(k) = \begin{bmatrix} i & x\,i & x^2 i & -\dot x & 1 & x & x^2 & x^3 \end{bmatrix}_k^\top, \qquad \theta = \begin{bmatrix} k_0 & k_1 & k_2 & c & a_0 & a_1 & a_2 & a_3 \end{bmatrix}^\top$$

**Why:** every unknown multiplies a signal that can be computed from data, so stage 1 becomes a linear least squares problem.

**Output:** the model structure and the parameter vector $\theta$ used in all later sections.

## 2. Simulator

### Plan
- Implement $f(\xi, u, \theta)$ and $h(\xi, u, \theta)$ from section 1.
- Integrate with a fixed step RK4 at $T_s = 1/2000$ s, holding the current constant within each step.
- Test with datasheet values: step response, small signal resonance near 4.5 Hz, and a comparison with the existing chirp.

### Why
- The simulator is reused in three places: the pre-check of experiments (section 3), stage 2 (section 10) and validation (section 11).
- A fixed step gives smooth, reproducible results, which stage 2 needs for reliable gradients.

**Output:** a tested simulator function.

# Part B. Experiment design

Guiding argument: each unknown is only clearly visible under specific conditions.

| Condition | What dominates | What it reveals |
|---|---|---|
| At rest | $K_f(x)\,i$ and $F_s(x)$ only | spring curve, $x_0$ |
| Low frequency | spring force | stiffness |
| Near resonance | damping | $c$ |
| High frequency | inertia, $m\ddot x \approx K_f i$ | $K_f$ (with $m$ known) |
| Different positions | local values | shape of $K_f(x)$ and $F_s(x)$ |

## 3. Signal design and simulated pre-check

### 3.1 Experiments to design

| Exp. | Name | Signal | Purpose |
|---|---|---|---|
| E0 | Sensor check | 30 s rest, then small sines at 10, 15, 20, 30 Hz | Noise levels, accelerometer gain $g_a$ |
| E1 | Static staircase | Current steps of 0.25 A, hold 2 to 3 s, up and down over both ends | Spring relation, $x_0$, current to position map, hysteresis |
| E2 | Local dynamics | DC current per position plus small periodic random phase multisine, 0.5 to 55 Hz, period 10 s | $K_f(\bar x)$, $k_{\text{eff}}(\bar x)$, $c$ at 5 to 7 positions |
| E3 | Identification data | Random phase multisine, 0.5 to 55 Hz, reduced low frequency amplitude, three levels (30%, 60%, 90% of stroke) | Main data for stages 1 and 2 |
| E4 | Validation data | Existing chirp, sines at 2, 4.5, 15 Hz (low and high amplitude), current steps | Independent test only |

### 3.2 Multisine definition

$$\tilde i(t) = \sum_k A_k \cos\!\left(2\pi k f_0\, t + \varphi_k\right), \qquad \varphi_k \text{ random in } [0, 2\pi)$$

### 3.3 Design arguments
- Periodic signals with an integer number of periods avoid spectral leakage, and averaging over periods reduces noise.
- Random phases avoid all sines peaking together, giving more excitation for the same peak current.
- A multisine produces many combinations of position, velocity and current, so the stage 1 regressors are less correlated than with a chirp.
- Several amplitude levels make sure every part of the stroke is covered with data.
- The system is well damped, $\zeta \approx 0.4$, so a 0.1 Hz frequency spacing and holds of 2 to 3 s are sufficient.

### 3.4 Simulated pre-check
- Run every profile through the simulator with datasheet values.
- Check peak position against the hard limit (a few mm inside the measured ends) and peak current against the drive limit.

**Output:** CSV current profiles at 2 kHz with safe amplitudes.

# Part C. Measurement

## 4. Experiments and data loading

### 4.1 Running the experiments
- Order: E0, E1, E2, E3, E4. E1 provides the DC currents for E2 and the amplitude limits for E3.
- Log at 2 kHz: time, measured and target current, position, accelerometer, coil temperature if available.
- Set the hard position limit in the drive before E3.
- Keep runs short and allow cooling, because magnet strength (and thus $K_f$) drops slightly with temperature.

### 4.2 Data loading and preprocessing
- Load each file, strip header spaces, check the sample rate, drop duplicate time stamps.
- Convert units (position to m, accelerometer with $g_a$ once known).
- Trim settling transients.
- Apply the **same** zero phase low pass filter to all signals, then differentiate with a Savitzky Golay filter to obtain $\dot x$ and $\ddot x$.

**Why identical filtering:** if only $x$ were filtered, its derivatives would be delayed relative to $i$, and the equation of motion would no longer hold even with the correct parameters.

**Output:** clean signals $x$, $\dot x$, $\ddot x$, $i$ per experiment.

# Part D. Identification

Two independent routes lead to the parameters:
- **Route 1** (sections 6 to 8): targeted experiments, each isolating one effect.
- **Route 2** (sections 9 and 10): rich data, all parameters fitted at once.

If both routes agree, the result can be trusted. If they disagree, the difference points to the cause.

## 5. Sensor calibration (E0)

### Plan
- From the rest record: $\sigma_x$, $\sigma_a$ and sensor offsets.
- From the sines: accelerometer gain at each frequency,

$$g_a = \frac{A_{\text{meas}}}{(2\pi f)^2 X}$$

### Why
- $\sigma_x$ and $\sigma_a$ become the weights of the stage 2 cost.
- In the 10 to 30 Hz range both sensors give large, accurate signals.

### Checkpoint
$g_a$ must be constant over frequency. If not, the sensor or its mounting has its own dynamics, and the analysis continues with position only.

**Output:** $\sigma_x$, $\sigma_a$, $g_a$, decision on using the accelerometer.

## 6. Static identification (E1)

### Plan
- Average position and current at the end of each hold.
- At rest $\dot x = \ddot x = 0$, so

$$F_s(\bar x) = -K_f(\bar x)\,\bar i$$

- The point at $\bar i = 0$ gives $x_0$.
- Build the current to position map.

### Why
This isolates the spring and motor force from all dynamic effects. On its own it cannot separate $K_f$ from $F_s$; section 7 resolves this.

### Checkpoint
Upward and downward sweeps must agree. A difference indicates friction or hysteresis not included in the model.

**Output:** static points $(\bar x, \bar i)$, $x_0$, current to position map.

## 7. Local linear identification (E2)

### Plan
- Compute the FRF from current to position at each position by averaging DFTs over periods.
- Fit the local second order model:

$$G(j\omega) = \frac{K_f(\bar x)}{k_{\text{eff}}(\bar x) - m\omega^2 + j c\,\omega}, \qquad k_{\text{eff}}(\bar x) = -F_s'(\bar x) - K_f'(\bar x)\,\bar i$$

### Why
Around a fixed position with small motion the model is locally linear. Each parameter appears in a different part of the FRF:
- high frequency: $\omega^2 |G| \rightarrow K_f(\bar x)/m$, which gives $K_f(\bar x)$ since $m$ is known
- low frequency: $G(0) = K_f/k_{\text{eff}}$, which gives the local stiffness
- resonance peak: gives $c$

**Output:** $K_f(\bar x)$, $k_{\text{eff}}(\bar x)$ and $c$ at each position, with uncertainties from the period to period variation.

## 8. First complete model

### Plan
- Fit the $K_f(x)$ polynomial to the points from section 7.
- Use it in the static relation from section 6 to get spring force points, and fit the $F_s(x)$ polynomial.
- Take $c$ from section 7.

### Checkpoint
The slope of the static spring curve must match the local stiffness from section 7 through $k_{\text{eff}} = -F_s' - K_f'\,\bar i$. Two independent experiments predicting the same quantity is a strong consistency test. A mismatch points to a calibration error or a missing effect, for example current dependent $K_f$ through saturation.

**Output:** initial parameter vector $\theta_0$.

## 9. Stage 1, equation error least squares (E3)

### Plan
- Build one regression row per sample using the preprocessed E3 signals, and add the static points from section 6 as extra rows with $\dot x = \ddot x = 0$.
- Scale $x$ so the columns of $\Phi$ have similar magnitudes, and check the condition number.
- Solve

$$\hat\theta_1 = \arg\min_\theta \|Y - \Phi\theta\|_2^2$$

### Why
- The model is linear in $\theta$, so there is a single solution, without initial guess or iteration.
- The result is slightly biased because $\dot x$ is derived from noisy data. Noise in a regressor pulls the estimate towards zero:

$$\hat c \rightarrow c\,\frac{\sigma_v^2}{\sigma_v^2 + \sigma_n^2}$$

  This is acceptable, since stage 1 only needs to provide a good starting point.

### Checkpoints
- Compare $\hat\theta_1$ with $\theta_0$ from section 8.
- Plot the force residual against $x$ and $\dot x$. A visible pattern means a missing term or too low a polynomial order.

**Output:** $\hat\theta_1$, the starting point for stage 2.

## 10. Stage 2, output error refinement (E3)

### Plan
- Parameters: $\theta$, initial states $\hat\xi(0)$ per record, and $g_a$ if the accelerometer is used.
- Simulate with the section 2 simulator, driven only by the measured current.
- Minimize the weighted cost over all three E3 levels together:

$$J(\theta) = \sum_k \left[\frac{\big(x(k) - \hat x(k,\theta)\big)^2}{\sigma_x^2} + \frac{\big(a(k) - \hat a(k,\theta)\big)^2}{\sigma_a^2}\right]$$

- Optimize with Levenberg Marquardt, starting from $\hat\theta_1$. If it stalls, fit a short segment first and extend gradually.
- Compute the parameter covariance $\operatorname{cov}(\hat\theta) \approx \hat\sigma^2 (J^\top J)^{-1}$.

### Why
- The simulation sees only the clean measured current, so sensor noise causes scatter but no systematic bias.
- It judges the model on accumulated behaviour, which is what a controller depends on.
- The cost is nonlinear and can have local minima, which is why stage 1 provides the start.
- $g_a$ is identifiable here because the position output fixes the true scale of the motion.

### Checkpoints
- Large variances or strong correlations in the covariance mean a polynomial order is too high or E3 lacks information.
- Compare with $\hat\theta_1$ (size of stage 1 bias) and $g_a$ with section 5.

**Output:** final parameters $\hat\theta$ with uncertainties.

# Part E. Validation and use

## 11. Validation (E4)

### Plan
- Simulate the chirp, sines and steps from the measured current alone.
- Compare simulated and measured position and acceleration, in time and as a fit percentage per record.
- Residual tests: auto correlation of the residuals, and cross correlation of residuals with the current.

### Why
This is the only test of whether the model generalizes beyond the data it was fitted on. E4 is never used for any fitting choice.

### Interpreting failures
| Symptom | Likely cause |
|---|---|
| Good at small amplitude, poor at large amplitude | Polynomial order too low, or E3 did not cover the ends of the stroke |
| Poor near resonance | Damping depends on position or velocity |
| Poor on steps only | Unmodelled input dynamics, such as the current loop |
| Residuals correlate with current | An input driven effect is missing |

**Output:** a validated model, or a diagnosis of what to improve.

## 12. Results and export

### Plan
- Table of final parameter values with uncertainties, compared with datasheet values.
- Plots of $K_f(x)$ and $F_s(x)$ over the measured range, with the points from sections 6 and 7 overlaid.
- Export the nonlinear model and linearizations at chosen operating points for controller design.

### Summary of decision points

| Section | Check | If it fails |
|---|---|---|
| 5 | $g_a$ constant over frequency | Use position only |
| 6 | Up and down sweeps agree | Add a friction term |
| 8 | Static slope matches local stiffness | Check calibration, consider current dependent $K_f$ |
| 9 | Residual force shows no pattern | Raise polynomial order or add a term |
| 10 | Parameters well determined | Lower polynomial order or improve E3 |
| 11 | Validation fit and residual tests | Interpret by symptom table |

**Output:** the model in a form ready for controller design.